In [1]:
import numpy as np
import pandas as pd

# ============================================================
# METHOD 1: Simple Loop (Most Readable)
# ============================================================
def create_sliding_window_simple(data, window_size):
    """
    Create sliding windows using a simple loop.
    
    Args:
        data: List or array of values
        window_size: Number of time steps to look back
    
    Returns:
        X: Input sequences (windows)
        y: Target values (next step after each window)
    """
    X, y = [], []
    
    for i in range(len(data) - window_size):
        # Get window of data
        window = data[i:i + window_size]
        # Get target (next value after window)
        target = data[i + window_size]
        
        X.append(window)
        y.append(target)
    
    return np.array(X), np.array(y)


# ============================================================
# METHOD 2: List Comprehension (Compact)
# ============================================================
def create_sliding_window_compact(data, window_size):
    """
    Create sliding windows using list comprehension.
    More Pythonic and faster than loops.
    """
    X = [data[i:i + window_size] for i in range(len(data) - window_size)]
    y = [data[i + window_size] for i in range(len(data) - window_size)]
    
    return np.array(X), np.array(y)


# ============================================================
# METHOD 3: NumPy Vectorized (Fastest)
# ============================================================
def create_sliding_window_numpy(data, window_size):
    """
    Create sliding windows using NumPy for maximum performance.
    Best for large datasets.
    """
    data = np.array(data)
    n_samples = len(data) - window_size
    
    # Create indices for all windows at once
    indices = np.arange(window_size)[None, :] + np.arange(n_samples)[:, None]
    
    X = data[indices]
    y = data[window_size:]
    
    return X, y


# ============================================================
# METHOD 4: Multi-Step Forecast (Predict Multiple Steps Ahead)
# ============================================================
def create_sliding_window_multistep(data, window_size, forecast_steps=1):
    """
    Create sliding windows for multi-step ahead forecasting.
    
    Args:
        data: Input data
        window_size: Number of past steps to use
        forecast_steps: Number of future steps to predict
    
    Returns:
        X: Input sequences
        y: Target sequences (multiple future values)
    """
    X, y = [], []
    
    for i in range(len(data) - window_size - forecast_steps + 1):
        window = data[i:i + window_size]
        targets = data[i + window_size:i + window_size + forecast_steps]
        
        X.append(window)
        y.append(targets)
    
    return np.array(X), np.array(y)


# ============================================================
# METHOD 5: With Stride/Step (Skip Data Points)
# ============================================================
def create_sliding_window_stride(data, window_size, step=1):
    """
    Create sliding windows with custom stride/step.
    
    Args:
        data: Input data
        window_size: Size of each window
        step: How many positions to move window each time (default=1)
    
    Example:
        step=1: Windows overlap completely
        step=window_size: No overlap (sequential chunks)
    """
    X, y = [], []
    
    for i in range(0, len(data) - window_size, step):
        window = data[i:i + window_size]
        target = data[i + window_size]
        
        X.append(window)
        y.append(target)
    
    return np.array(X), np.array(y)


# ============================================================
# METHOD 6: For LSTM (3D Shape with Features)
# ============================================================
def create_sliding_window_lstm(data, window_size, n_features=1):
    """
    Create sliding windows in LSTM format: (samples, timesteps, features)
    
    Args:
        data: Input data (can be 1D or 2D)
        window_size: Number of time steps
        n_features: Number of features per timestep
    
    Returns:
        X: Shape (samples, window_size, n_features)
        y: Shape (samples,)
    """
    data = np.array(data)
    
    # Handle 1D data
    if data.ndim == 1:
        data = data.reshape(-1, 1)
    
    X, y = [], []
    
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size, 0])  # Predict first feature
    
    X = np.array(X)
    y = np.array(y)
    
    return X, y


# ============================================================
# METHOD 7: Train-Test Split with Windows
# ============================================================
def create_train_test_windows(data, window_size, train_ratio=0.8):
    """
    Create sliding windows and split into train/test sets.
    
    Args:
        data: Input data
        window_size: Size of lookback window
        train_ratio: Proportion of data for training
    
    Returns:
        X_train, y_train, X_test, y_test
    """
    X, y = create_sliding_window_simple(data, window_size)
    
    # Split based on time (important for time series!)
    split_idx = int(len(X) * train_ratio)
    
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    return X_train, y_train, X_test, y_test


# ============================================================
# EXAMPLE USAGE & DEMONSTRATION
# ============================================================
if __name__ == "__main__":
    # Generate sample temperature data
    temperatures = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 
                   30, 31, 32, 31, 30, 29, 28, 27, 26, 25]
    
    print("="*60)
    print("ORIGINAL DATA:")
    print(temperatures)
    print(f"Length: {len(temperatures)} days")
    
    # Example 1: Simple 5-day window
    print("\n" + "="*60)
    print("EXAMPLE 1: Simple 5-Day Window")
    print("="*60)
    X, y = create_sliding_window_simple(temperatures, window_size=5)
    print(f"Number of samples: {len(X)}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print("\nFirst 3 samples:")
    for i in range(3):
        print(f"Sample {i+1}:")
        print(f"  Input (X):  {X[i]} → Output (y): {y[i]}")
    
    # Example 2: Multi-step forecast
    print("\n" + "="*60)
    print("EXAMPLE 2: Multi-Step Forecast (Predict 3 days ahead)")
    print("="*60)
    X, y = create_sliding_window_multistep(temperatures, window_size=5, forecast_steps=3)
    print(f"Number of samples: {len(X)}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print("\nFirst sample:")
    print(f"  Input (X):  {X[0]}")
    print(f"  Output (y): {y[0]} (next 3 days)")
    
    # Example 3: LSTM format
    print("\n" + "="*60)
    print("EXAMPLE 3: LSTM Format (3D)")
    print("="*60)
    X, y = create_sliding_window_lstm(temperatures, window_size=5)
    print(f"X shape: {X.shape} (samples, timesteps, features)")
    print(f"y shape: {y.shape}")
    
    # Example 4: With stride
    print("\n" + "="*60)
    print("EXAMPLE 4: With Stride=3 (Less Overlap)")
    print("="*60)
    X, y = create_sliding_window_stride(temperatures, window_size=5, step=3)
    print(f"Number of samples: {len(X)} (fewer due to stride)")
    print("\nAll samples:")
    for i in range(len(X)):
        print(f"Sample {i+1}: {X[i]} → {y[i]}")
    
    # Performance comparison
    print("\n" + "="*60)
    print("PERFORMANCE COMPARISON (Large Dataset)")
    print("="*60)
    large_data = np.random.randn(10000)
    
    import time
    
    methods = [
        ("Simple Loop", create_sliding_window_simple),
        ("List Comprehension", create_sliding_window_compact),
        ("NumPy Vectorized", create_sliding_window_numpy)
    ]
    
    for name, func in methods:
        start = time.time()
        X, y = func(large_data, window_size=30)
        elapsed = time.time() - start
        print(f"{name:20s}: {elapsed:.4f} seconds | Shape: {X.shape}")
    
    print("\n" + "="*60)
    print("✓ All examples completed successfully!")
    print("="*60)

ORIGINAL DATA:
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 31, 30, 29, 28, 27, 26, 25]
Length: 20 days

EXAMPLE 1: Simple 5-Day Window
Number of samples: 15
X shape: (15, 5)
y shape: (15,)

First 3 samples:
Sample 1:
  Input (X):  [20 21 22 23 24] → Output (y): 25
Sample 2:
  Input (X):  [21 22 23 24 25] → Output (y): 26
Sample 3:
  Input (X):  [22 23 24 25 26] → Output (y): 27

EXAMPLE 2: Multi-Step Forecast (Predict 3 days ahead)
Number of samples: 13
X shape: (13, 5)
y shape: (13, 3)

First sample:
  Input (X):  [20 21 22 23 24]
  Output (y): [25 26 27] (next 3 days)

EXAMPLE 3: LSTM Format (3D)
X shape: (15, 5, 1) (samples, timesteps, features)
y shape: (15,)

EXAMPLE 4: With Stride=3 (Less Overlap)
Number of samples: 5 (fewer due to stride)

All samples:
Sample 1: [20 21 22 23 24] → 25
Sample 2: [23 24 25 26 27] → 28
Sample 3: [26 27 28 29 30] → 31
Sample 4: [29 30 31 32 31] → 30
Sample 5: [32 31 30 29 28] → 27

PERFORMANCE COMPARISON (Large Dataset)
Simple Loop         :